<a href="https://colab.research.google.com/github/irene501/flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/irene501/flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [ ]:


print("Task type: Binary classification with probability-based ranking.")
print()

print("The main goal is to predict whether a page is declining or not.")
print("This makes it a binary classification problem because there are only two possible outcomes:")
print("declining (1) or not declining (0).")
print()

print("Instead of only predicting yes or no, the model also produces a probability score.")
print("This score is used to rank pages from the highest risk of decline to the lowest.")
print()

print("It is not a ranking task because the model does not compare pages directly with one another.")
print("It is also not a clustering task because it does not group similar pages together.")
print("Each page is evaluated individually and assigned a probability of decline./n")

import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/irene501/flyrank/main/data/raw/content_refresh_anonymized.csv")
# The task in one line: for each page, is trend_direction == "down"?
print()
print(df["trend_direction"].value_counts())
print(f"\n'down' share of all pages: {(df['trend_direction'] == 'down').mean():.1%}")


Task type: Binary classification with probability-based ranking.

The main goal is to predict whether a page is declining or not.
This makes it a binary classification problem because there are only two possible outcomes:
declining (1) or not declining (0).

Instead of only predicting yes or no, the model also produces a probability score.
This score is used to rank pages from the highest risk of decline to the lowest.

It is not a ranking task because the model does not compare pages directly with one another.
It is also not a clustering task because it does not group similar pages together.
Each page is evaluated individually and assigned a probability of decline./n

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

'down' share of all pages: 54.2%


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
print("Target or Proxy")
print()

print("Target: is_declining_label (a proxy, not a future outcome).")
print()
print("The target is created using the rule:")
print("is_declining_label = 1 if trend_direction == 'down' else 0.")
print()
print("The trend_direction value is calculated by comparing the last 30 days")
print("of traffic with the previous 30 days within the same 90-day period.")
print()
print("This means the label describes the page's current condition rather than")
print("predicting what will happen in the future.")
print()
print("It is a useful proxy for identifying pages that may need attention,")
print("but it should not be described as a forecast.")
print()
print("A better version of the project would use a future-looking target,")
print("such as whether the page declines in the 30 days after the feature")
print("window, allowing the model to predict future outcomes instead of")
print("describing the present.")
print()
#How the proxy is actually built
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print()
print(f"declining-label rate: {df['is_declining_label'].mean():.3f}")
print(f"declining-label rows: {df['is_declining_label'].sum():,} of {len(df):,}")


Target or Proxy

Target: is_declining_label (a proxy, not a future outcome).

The target is created using the rule:
is_declining_label = 1 if trend_direction == 'down' else 0.

The trend_direction value is calculated by comparing the last 30 days
of traffic with the previous 30 days within the same 90-day period.

This means the label describes the page's current condition rather than
predicting what will happen in the future.

It is a useful proxy for identifying pages that may need attention,
but it should not be described as a forecast.

A better version of the project would use a future-looking target,
such as whether the page declines in the 30 days after the feature
window, allowing the model to predict future outcomes instead of
describing the present.


declining-label rate: 0.542
declining-label rows: 16,262 of 30,000


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:


print("Metric: Precision@50.")
print()

print("Precision@50 measures how many of the top 50 pages ranked by the model")
print("are actually declining pages.")
print()

print("This metric is more suitable than accuracy or ROC AUC because reviewers")
print("only focus on the highest-ranked pages rather than the entire list.")
print()

print("The goal is to ensure that the pages at the top of the ranking are")
print("the ones most likely to need attention.")
print()

print("In this scenario, precision is more important than recall because")
print("reviewers have limited time. Missing a page ranked much lower in the")
print("list is less costly than placing a non-declining page near the top,")
print("where it would waste the reviewer's time.")
print()
# Why "top of the queue" is the part that matters
total_pages = len(df)
declining_pages = int(df["is_declining_label"].sum())
reviewer_capacity_per_week = 50  # a rough, defensible weekly review capacity

print(f"total pages in inventory: {total_pages:,}")
print(f"pages flagged declining:  {declining_pages:,} ({declining_pages/total_pages:.1%} of inventory)")
print(f"a reviewer can realistically get through ~{reviewer_capacity_per_week} pages/week")
print(f"-> that's {reviewer_capacity_per_week/total_pages:.2%} of the inventory, so precision at that exact cutoff is what matters")

Metric: Precision@50.

Precision@50 measures how many of the top 50 pages ranked by the model
are actually declining pages.

This metric is more suitable than accuracy or ROC AUC because reviewers
only focus on the highest-ranked pages rather than the entire list.

The goal is to ensure that the pages at the top of the ranking are
the ones most likely to need attention.

In this scenario, precision is more important than recall because
reviewers have limited time. Missing a page ranked much lower in the
list is less costly than placing a non-declining page near the top,
where it would waste the reviewer's time.

total pages in inventory: 30,000
pages flagged declining:  16,262 (54.2% of inventory)
a reviewer can realistically get through ~50 pages/week
-> that's 0.17% of the inventory, so precision at that exact cutoff is what matters


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
# One row per page: content_id is the unit of analysis
unit_cols = [
    "content_id", "client_id", "content_type", "content_age_days",
    "impressions_90d", "clicks_90d", "avg_position", "ctr",
    "trend_direction", "is_declining_label",
]

df[unit_cols].head(8)

,content_id,client_id,content_type,content_age_days,impressions_90d,clicks_90d,avg_position,ctr,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,187,3803,29,10.6,0.76,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,15320,7,20.3,0.05,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,12581,11,36.5,0.09,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,463,11751,58,6.2,0.49,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,263,19140,24,44.0,0.13,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,147,3970,1,8.5,0.03,down,1
6,content_9a34b442b552,client_8722616204,keyword article,90,20,0,7.0,0.00,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,445,1724,1,21.2,0.06,stable,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:


print("A fixed rule can only use one condition at a time, such as flagging")
print("a page because it is old or because its impressions have dropped.")
print()

print("However, a page that is declining and worth reviewing usually depends")
print("on several factors working together, such as impressions, search")
print("position, engagement, and page age.")
print()

print("Machine learning can learn how these features interact instead of")
print("relying on a single threshold or rule.")
print()

print("The project's baseline rule achieved a Precision@50 score of 0.240,")
print("meaning only about 12 of the top 50 recommended pages were actually")
print("declining.")
print()

print("A Random Forest model trained on the same features achieved a")
print("Precision@50 score of 0.740 on unseen client data, correctly")
print("identifying about 37 of the top 50 pages.")
print()

print("This shows that machine learning is more effective because it learns")
print("patterns from multiple features instead of depending on manually")
print("chosen rules.")
print()
# Verified numbers from outputs/model_report.md (client-holdout validation)
comparison = pd.DataFrame([
    {"method": "baseline rules",       "roc_auc": 0.627, "precision_at_50": 0.240},
    {"method": "logistic regression",  "roc_auc": 0.700, "precision_at_50": 0.400},
    {"method": "decision tree",        "roc_auc": 0.742, "precision_at_50": 0.540},
    {"method": "random forest",        "roc_auc": 0.750, "precision_at_50": 0.740},
])
comparison["correct_in_top_50"] = (comparison["precision_at_50"] * 50).round().astype(int)
comparison

A fixed rule can only use one condition at a time, such as flagging
a page because it is old or because its impressions have dropped.

However, a page that is declining and worth reviewing usually depends
on several factors working together, such as impressions, search
position, engagement, and page age.

Machine learning can learn how these features interact instead of
relying on a single threshold or rule.

The project's baseline rule achieved a Precision@50 score of 0.240,
meaning only about 12 of the top 50 recommended pages were actually
declining.

A Random Forest model trained on the same features achieved a
Precision@50 score of 0.740 on unseen client data, correctly
identifying about 37 of the top 50 pages.

This shows that machine learning is more effective because it learns
patterns from multiple features instead of depending on manually
chosen rules.



,method,roc_auc,precision_at_50,correct_in_top_50
0,baseline rules,0.627,0.24,12
1,logistic regression,0.700,0.40,20
2,decision tree,0.742,0.54,27
3,random forest,0.750,0.74,37


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.